# 🛑 Phone-Robot: 単眼深度推定（Depth Anything / MiDaS）スマホ軽量化 & TFLite変換パイプライン

このノートブックは、スマホ（Galaxy Note 10+）の単眼カメラ1枚から **「前方の壁・障害物への接近度合い（相対深度）」をリアルタイム推定し、自動ブレーキをかける** ための軽量深度モデルを構築・TFLite量子化する完全自動パイプラインです。

### 📋 なぜこれが必要か
- トイラジコンに高価なLiDARや超音波センサーを増設しなくても、**スマホのカメラ映像だけで「前方に障害物があるか」を判定可能**になります。
- Depth Anything V2 Small / MiDaS Small の重みをモバイルGPU向け（FP16）に量子化し、Note 10+上で30FPS以上のリアルタイム推論を目指します。


In [ ]:
# 1. 必要なライブラリのインストールとGPU確認
!nvidia-smi

!pip install -q timm onnx onnxsim onnx2tf tensorflow tf-keras

import os
import time
import urllib.request
import numpy as np
import matplotlib.pyplot as plt
import cv2
import torch
import torch.nn as nn
import tensorflow as tf

print(f"PyTorch Version: {torch.__version__}, CUDA Available: {torch.cuda.is_available()}")
print(f"TensorFlow Version: {tf.__version__}")


In [ ]:
# 2. スマホ向け最軽量深度推定モデル（MiDaS v2.1 Small）の読み込み
# Note 10+で爆速動作する実績のある超軽量バックボーン（約21MB -> 量子化後約10MB）
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

print("TorchHubからMiDaS_smallモデルをロードしています...")
midas = torch.hub.load("intel-isl/MiDaS", "MiDaS_small", pretrained=True)
midas.to(device)
midas.eval()
print("モデルロード完了!")


In [ ]:
# 3. テスト画像での相対深度推定テスト
# 室内・道路のテスト画像をダウンロード
test_img_url = "https://raw.githubusercontent.com/intel-isl/MiDaS/master/input/dog.jpg"
test_img_path = "test_image.jpg"
if not os.path.exists(test_img_path):
    urllib.request.urlretrieve(test_img_url, test_img_path)

img_bgr = cv2.imread(test_img_path)
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

# スマホ向け固定解像度 (256x256)
INPUT_SIZE = 256
img_resized = cv2.resize(img_rgb, (INPUT_SIZE, INPUT_SIZE))
img_input = img_resized.astype(np.float32) / 255.0
# ImageNet正規化
mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
img_input = (img_input - mean) / std
img_tensor = torch.from_numpy(img_input).permute(2, 0, 1).unsqueeze(0).to(device)

with torch.no_grad():
    prediction = midas(img_tensor)
    depth_map = prediction.squeeze().cpu().numpy()

# 0〜1に正規化
depth_norm = (depth_map - depth_map.min()) / (depth_map.max() - depth_map.min() + 1e-8)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(img_resized)
plt.title(f"Input Image ({INPUT_SIZE}x{INPUT_SIZE})")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(depth_norm, cmap='inferno')
plt.title("Relative Depth Map (Brighter = Closer)")
plt.axis('off')
plt.colorbar()
plt.tight_layout()
plt.show()


In [ ]:
# 4. 「自動ブレーキ」判定ロジックの実装
# 画面中央下部（ラジコンの直進進路上）の近接度をサンプリング
def evaluate_collision_risk(depth_normalized, threshold=0.70):
    h, w = depth_normalized.shape
    # 中央下部（進路上のエリア）
    roi = depth_normalized[int(h*0.50):int(h*0.90), int(w*0.35):int(w*0.65)]
    risk_score = float(np.mean(roi))
    should_brake = risk_score > threshold
    return risk_score, should_brake

risk, brake = evaluate_collision_risk(depth_norm)
print(f"進路上障害物リスクスコア: {risk:.2f} (0.0=遠い, 1.0=極めて近い)")
print(f"ブレーキ指令: {'🛑 緊急停止 (BRAKE)' if brake else '🟢 走行継続 (CLEAR)'}")


In [ ]:
# 5. ONNX へのエクスポート
os.makedirs("models", exist_ok=True)
onnx_path = "models/midas_small.onnx"

dummy_input = torch.randn(1, 3, INPUT_SIZE, INPUT_SIZE, device=device)
print(f"ONNXエクスポート中: {onnx_path} ...")
torch.onnx.export(
    midas,
    dummy_input,
    onnx_path,
    input_names=["input_image"],
    output_names=["relative_depth"],
    opset_version=13,
    do_constant_folding=True
)
print(f"ONNX出力成功! サイズ: {os.path.getsize(onnx_path) / (1024*1024):.2f} MB")


In [ ]:
# 6. TensorFlow Lite (FP16) への変換
# onnx2tf を用いた最適化変換
print("ONNXからTFLiteへ変換しています...")
!onnx2tf -in models/midas_small.onnx -coi -output_signaturedefs

# 生成されたSavedModelからTFLiteへコンパイル
saved_model_dir = "saved_model"
converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_dir)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
tflite_fp16 = converter.convert()

tflite_path = "models/model_depth_small_fp16.tflite"
with open(tflite_path, "wb") as f:
    f.write(tflite_fp16)

file_size_mb = os.path.getsize(tflite_path) / (1024*1024)
print(f"=== TFLite (FP16) エクスポート成功! ===")
print(f"ファイル名: {tflite_path}")
print(f"モデルサイズ: {file_size_mb:.2f} MB (モバイルGPUに最適)")


In [ ]:
# 7. スマホ向け推論速度ベンチマーク
interpreter = tf.lite.Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()
print(f"TFLite Input: {input_details[0]['shape']}, Output: {output_details[0]['shape']}")

dummy_tflite_input = np.random.randn(*input_details[0]['shape']).astype(input_details[0]['dtype'])

# ウォームアップ
for _ in range(5):
    interpreter.set_tensor(input_details[0]['index'], dummy_tflite_input)
    interpreter.invoke()

# 50回計測
latencies = []
for _ in range(50):
    t0 = time.perf_counter()
    interpreter.set_tensor(input_details[0]['index'], dummy_tflite_input)
    interpreter.invoke()
    latencies.append((time.perf_counter() - t0) * 1000.0)

mean_ms = np.mean(latencies)
print(f"平均推論時間: {mean_ms:.2f} ms")
print(f"推定FPS: {1000.0 / mean_ms:.1f} FPS")


In [ ]:
# モデルのダウンロード (Colab環境なら自動ダウンロード、ローカルなら保存のみ)
try:
    from google.colab import files
    print('生成されたモデルファイルをダウンロードします...')
    target_file = tflite_path if 'tflite_path' in locals() else 'models/model_e2e_fp16.tflite'
    files.download(target_file)
except ImportError:
    print('ローカル環境で実行中: モデルファイルは models/ ディレクトリに正常保存されました。')
